In [8]:
experiment = "vggnet16_10_ten_imgs"

In [9]:
# To sort the results.csv by instance_id
CSV_PATH = f"../results/{experiment}/results.csv"

import pandas as pd
df = pd.read_csv(CSV_PATH)
df["instance_id"] = pd.to_numeric(df["instance_id"], errors="coerce")
df_sorted = df.sort_values(by="instance_id", ascending=True)
df_sorted.to_csv(CSV_PATH, index=False)

In [10]:
import pandas as pd
import re
from pathlib import Path

results_path = f"../results/{experiment}/results.csv"
stats_path = f"../results/{experiment}/input_change_stats.csv"
out_path = f"../results/{experiment}/combined_results.csv"


df_res = pd.read_csv(results_path)
df_ics = pd.read_csv(stats_path)

# Accept BOTH:
# 1) image_global_kK_eps_E.vnnlib
# 2) image_seg0_fixmask_kK_eps_E.vnnlib
# 3) image_fix_mask_seg0_kK_eps_E.vnnlib
# plus fixnonmask / fix_nonmask variants
RX_GLOBAL = re.compile(r"""(?x)
(?:.*/)?(?P<image>.+?)_global_k(?P<k>\d+)_eps_(?P<eps>[0-9]*\.?[0-9]+)\.vnnlib$
""")

RX_SEG_A = re.compile(r"""(?x)
(?:.*/)?(?P<image>.+?)_seg(?P<seg>\d+)_(?P<tag>fixmask|fixnonmask|fix_mask|fix_nonmask)
_k(?P<k>\d+)_eps_(?P<eps>[0-9]*\.?[0-9]+)\.vnnlib$
""")

RX_SEG_B = re.compile(r"""(?x)
(?:.*/)?(?P<image>.+?)_(?P<tag>fixmask|fixnonmask|fix_mask|fix_nonmask)_seg(?P<seg>\d+)
_k(?P<k>\d+)_eps_(?P<eps>[0-9]*\.?[0-9]+)\.vnnlib$
""")

def normalize_tag(tag: str) -> str:
    tag = tag.lower()
    tag = tag.replace("fixmask", "fix_mask").replace("fixnonmask", "fix_nonmask")
    return tag

def parse_vnnlib(vnnlib_path: str):
    s = str(vnnlib_path)

    m = RX_GLOBAL.search(s)
    if m:
        return pd.Series({
            "image": m.group("image"),
            "tag": "global",
            "segment_index": -1,
            "k": int(m.group("k")),
            "eps": float(m.group("eps")),
        })

    for rx in (RX_SEG_A, RX_SEG_B):
        m = rx.search(s)
        if m:
            return pd.Series({
                "image": m.group("image"),
                "tag": normalize_tag(m.group("tag")),
                "segment_index": int(m.group("seg")),
                "k": int(m.group("k")),
                "eps": float(m.group("eps")),
            })

    # If something doesn't match, keep it visible (so you can debug)
    return pd.Series({"image": None, "tag": None, "segment_index": None, "k": None, "eps": None})

# Parse keys from results.csv
parsed = df_res["vnnlib"].apply(parse_vnnlib)
df_res2 = pd.concat([df_res, parsed], axis=1)

# Extract model name from onnx path (onnx/vgg16-7.onnx -> vgg16-7)
df_res2["model"] = df_res2["onnx"].astype(str).apply(lambda p: Path(p).stem)

# Ensure types match (important for merging!)
df_ics2 = df_ics.copy()
df_ics2["k"] = df_ics2["k"].astype("int64")
df_ics2["segment_index"] = df_ics2["segment_index"].astype("int64")
df_ics2["eps"] = df_ics2["eps"].astype("float64")
df_ics2["model"] = df_ics2["model"].astype(str)

# Merge on the actual shared keys:
keys = ["image", "model", "eps", "k", "tag", "segment_index"]

merged = df_res2.merge(df_ics2, on=keys, how="left", suffixes=("_res", "_ics"))

# Sanity checks
matched = merged["pattern"].notna().sum()  # 'pattern' exists in input_change_stats.csv
total = len(merged)
print(f"Matched rows: {matched}/{total}  ({matched/total:.3f})")

# Show any rows that failed to parse or match
bad_parse = merged[merged["image"].isna()][["vnnlib"]].head(20)
if len(bad_parse):
    print("\nExamples that FAILED TO PARSE (fix regex for these):")
    print(bad_parse.to_string(index=False))

bad_match = merged[merged["pattern"].isna()][["vnnlib"] + keys].head(20)
if len(bad_match):
    print("\nExamples that PARSED but DIDN'T MATCH stats keys:")
    print(bad_match.to_string(index=False))

merged.to_csv(out_path, index=False)
print("Wrote:", out_path)

Matched rows: 180/180  (1.000)
Wrote: ../results/vggnet16_10_ten_imgs/combined_results.csv


------

In [11]:
# To filter a dataframe to only images that appear exactly 3 times (G O B)
import pandas as pd
import numpy as np

def complete_triplets(df, image_col="image", debug_path=None):
    df = df.copy()
    print("Original shape:", df.shape)

    # keep only images that appear exactly 3 times
    counts = df[image_col].value_counts()
    good_images = counts[counts == 3].index
    df = df[df[image_col].isin(good_images)].copy()

    print("Number of images:", len(good_images))
    print("Shape after filtering to triplets:", df.shape)

    if debug_path is not None:
        df.to_csv(f"{debug_path}_{len(good_images)}_imgs.csv", index=False)

for i in range(1, 11): # always one more

    experiment = f"vggnet16_{i}_ten_imgs"
    CSV_PATH = f"../results/{experiment}/combined_results.csv"
    OUT_DIR  = f"../results/{experiment}"

    df = pd.read_csv(CSV_PATH)

    # numeric safety
    df["k"] = pd.to_numeric(df["k"], errors="coerce")
    df["eps"] = pd.to_numeric(df["eps"], errors="coerce")

    # (k, eps) -> subdf
    dfs_by_keps = {key: subdf.copy() for key, subdf in df.groupby(["k", "eps"])}

    # filter and save each (k, eps)
    dfs_by_keps_triplets = {}
    for (k, eps), subdf in dfs_by_keps.items():
        print(f"Processing k={k}, eps={eps}")
        dbg = f"{OUT_DIR}/triplets_k{k}_eps{eps}"
        complete_triplets(subdf, debug_path=dbg)

Processing k=1568, eps=0.0001
Original shape: (24, 24)
Number of images: 8
Shape after filtering to triplets: (24, 24)
Processing k=3136, eps=0.0001
Original shape: (24, 24)
Number of images: 8
Shape after filtering to triplets: (24, 24)
Processing k=6272, eps=0.0001
Original shape: (24, 24)
Number of images: 8
Shape after filtering to triplets: (24, 24)
Processing k=12544, eps=0.0001
Original shape: (24, 24)
Number of images: 8
Shape after filtering to triplets: (24, 24)
Processing k=25088, eps=0.0001
Original shape: (24, 24)
Number of images: 8
Shape after filtering to triplets: (24, 24)
Processing k=50176, eps=0.0001
Original shape: (24, 24)
Number of images: 8
Shape after filtering to triplets: (24, 24)
Processing k=1568, eps=0.0001
Original shape: (30, 24)
Number of images: 10
Shape after filtering to triplets: (30, 24)
Processing k=3136, eps=0.0001
Original shape: (30, 24)
Number of images: 10
Shape after filtering to triplets: (30, 24)
Processing k=6272, eps=0.0001
Original shap

In [12]:
# Merge all triplets*.csv from vggnet folders
from pathlib import Path
import pandas as pd

BASE_RESULTS = Path("../results")

OUTPUT_PATH = Path("../analysis/csvs/ALL_vggnet_triplets_merged.csv")

dfs = []

for subdir in BASE_RESULTS.iterdir():
    if subdir.is_dir() and subdir.name.startswith("vggnet"):
        print(f"\nEntering {subdir.name}")

        for csv_file in subdir.glob("triplets*.csv"):
            print("   ➜ taking:", csv_file.name)

            # read but skip header
            df = pd.read_csv(csv_file, header=0)

            dfs.append(df)

# concat everything
big_df = pd.concat(dfs, ignore_index=True)

# save
big_df.to_csv(OUTPUT_PATH, index=False)

print("\n=======================")
print(" Saved merged file to:")
print(OUTPUT_PATH)
print("Rows:", len(big_df))


Entering vggnet16_1_ten_imgs
   ➜ taking: triplets_k1568_eps0.0001_8_imgs.csv
   ➜ taking: triplets_k12544_eps0.0001_8_imgs.csv
   ➜ taking: triplets_k3136_eps0.0001_8_imgs.csv
   ➜ taking: triplets_k25088_eps0.0001_8_imgs.csv
   ➜ taking: triplets_k50176_eps0.0001_8_imgs.csv
   ➜ taking: triplets_k6272_eps0.0001_8_imgs.csv

Entering vggnet16_one_img_142
   ➜ taking: triplets_k50176_eps0.0001_1_imgs.csv
   ➜ taking: triplets_k6272_eps0.0001_1_imgs.csv
   ➜ taking: triplets_k1568_eps0.0001_1_imgs.csv
   ➜ taking: triplets_k12544_eps0.0001_1_imgs.csv
   ➜ taking: triplets_k3136_eps0.0001_1_imgs.csv
   ➜ taking: triplets_k25088_eps0.0001_1_imgs.csv

Entering vggnet16_7_ten_imgs
   ➜ taking: triplets_k6272_eps0.0001_7_imgs.csv
   ➜ taking: triplets_k50176_eps0.0001_7_imgs.csv
   ➜ taking: triplets_k12544_eps0.0001_7_imgs.csv
   ➜ taking: triplets_k3136_eps0.0001_7_imgs.csv
   ➜ taking: triplets_k25088_eps0.0001_7_imgs.csv
   ➜ taking: triplets_k1568_eps0.0001_7_imgs.csv

Entering vggnet16

------

In [ ]:
# To find BnB Instance (`domains_visited > 0`)
import os
import pandas as pd

experiment = "exp_1/triplets_k50176.0_eps0.0001_239_imgs"

CSV_PATH = f"results/{experiment}.csv"

df = pd.read_csv(CSV_PATH)

assert "domains_visited" in df.columns, "domains_visited column not found!"

df["domains_visited"] = pd.to_numeric(df["domains_visited"], errors="coerce").fillna(0)

bnb_df = df[df["domains_visited"] > 0].copy()

bnb_df = bnb_df.sort_values(
    by="lb_minus_rhs",
    ascending=False
)

print("Rows with domains_visited > 0:", len(bnb_df))
bnb_df.head()


cols = [
    "instance_id",
    "image",
    "tag",
    "is_global",
    "segment_index",
    "eps",
    "k",
    "result",
    "lb_minus_rhs",
    "domains_visited",
    "bab_time",
    "all_time",
]

# Only keep columns that exist (safe if your CSV is slightly different)
cols = [c for c in cols if c in bnb_df.columns]

bnb_df[cols].sort_values("domains_visited", ascending=False)

# OUT_PATH = f"rows_with_bnb_entered_{experiment}.csv"
# os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
# bnb_df.to_csv(OUT_PATH, index=False)
# print("Saved to", OUT_PATH)

In [ ]:
# To show the table
import csv

CSV = "csvs/all_vggnet16_results_k_all.csv"   # your merged file

with open(CSV, newline="") as f:
    r = csv.reader(f)
    rows = list(r)

# print as a simple aligned table
widths = [max(len(str(x)) for x in col) for col in zip(*rows)]
for i, row in enumerate(rows):
    line = " | ".join(str(x).ljust(w) for x, w in zip(row, widths))
    print(line)
    if i == 0:
        print("-+-".join("-"*w for w in widths))

vnnlib                                                     | timeout | result        | all_time           | onnx              | instance_id | lb_minus_rhs        | bab_time           | domains_visited | init_unstable | source_folder                               
-----------------------------------------------------------+---------+---------------+--------------------+-------------------+-------------+---------------------+--------------------+-----------------+---------------+---------------------------------------------
vnnlib/n02033041_dowitcher_global_k50176_eps_0.0001.vnnlib | 1200    | timeout False | 4005.6531410217285 | onnx/vgg16-7.onnx | 1           | -263.40545654296875 | 3988.4435873031616 | 3               | 0             | vggnet16_benchmark2022_one_img_naive        
vnnlib/n02033041_dowitcher_global_k50176_eps_0.0001.vnnlib | 1200    | timeout False | 4001.934016942978  | onnx/vgg16-7.onnx | 1           | -263.4098205566406  | 3984.5712988376617 | 3               | 0    

In [ ]:
# To show the table
import csv

CSV = "csvs/all_vggnet16_results_k_500.csv"   # your merged file

with open(CSV, newline="") as f:
    r = csv.reader(f)
    rows = list(r)

# print as a simple aligned table
widths = [max(len(str(x)) for x in col) for col in zip(*rows)]
for i, row in enumerate(rows):
    line = " | ".join(str(x).ljust(w) for x, w in zip(row, widths))
    print(line)
    if i == 0:
        print("-+-".join("-"*w for w in widths))

vnnlib                                                   | timeout | result      | all_time           | onnx              | instance_id | lb_minus_rhs       | bab_time           | domains_visited | init_unstable | source_folder                                    
---------------------------------------------------------+---------+-------------+--------------------+-------------------+-------------+--------------------+--------------------+-----------------+---------------+--------------------------------------------------
vnnlib/n02033041_dowitcher_global_k500_eps_0.0001.vnnlib | 1200    | unsat False | 842.1632499694824  | onnx/vgg16-7.onnx | 1           | 1.3206205368041992 | 794.1718530654907  | 0               | 0             | vggnet16_benchmark2022_one_img_naive_k500        
vnnlib/n02033041_dowitcher_global_k500_eps_0.0001.vnnlib | 1200    | unsat False | 804.6202256679535  | onnx/vgg16-7.onnx | 1           | 1.3206206560134888 | 793.8658721446991  | 0               | 0         

In [ ]:
# To show the table
import csv

CSV = "csvs/all_vggnet16_results_ks_12_14.csv"   # your merged file

with open(CSV, newline="") as f:
    r = csv.reader(f)
    rows = list(r)

# print as a simple aligned table
widths = [max(len(str(x)) for x in col) for col in zip(*rows)]
for i, row in enumerate(rows):
    line = " | ".join(str(x).ljust(w) for x, w in zip(row, widths))
    print(line)
    if i == 0:
        print("-+-".join("-"*w for w in widths))

vnnlib                                                              | domains_visited | init_unstable | result        | timeout | instance_id | lb_minus_rhs        | bab_time           | all_time           | onnx              | source_folder                                  
--------------------------------------------------------------------+-----------------+---------------+---------------+---------+-------------+---------------------+--------------------+--------------------+-------------------+------------------------------------------------
vnnlib/n02033041_dowitcher_global_k50176_eps_0.0001.vnnlib          |                 |               | error         | 1200    | 1           |                     |                    |                    | onnx/vgg16-7.onnx | vggnet16_benchmark2022_one_img_original        
vnnlib/n02033041_dowitcher_seg0_fixmask_k50176_eps_0.0001.vnnlib    | 0               | 0             | unsat False   | 1200    | 2           | 1.3113386631011963  | 885.98